# Kenning demo

[Kenning](https://huggingface.co/systemonedev/kenning-large-v0.4) is an open **System One** decision model (Apache-2.0): you ask typed questions about text or JSON (*yes/no*, *pick one*, *rate on a scale*) and get a calibrated probability for every option, in one pass, without generating text.

This notebook runs it on Colab's free GPU. **Runtime → Change runtime type → T4 GPU** first (CPU works too, slower), then **Runtime → Run all**.

Learn more at [systemone.dev](https://systemone.dev) · source: [SystemOne Builder](https://github.com/systemonedev/systemone-builder)

In [ ]:
%pip install -q "systemone-client[local]" gradio

## Ask in Python

The first run downloads the model (~0.9 GB).

In [ ]:
import torch
from systemone import Kenning, Noul, Choice, Score

device = "cuda" if torch.cuda.is_available() else "cpu"
model = Kenning.from_pretrained("systemonedev/kenning-large-v0.4", device=device)

r = model.system_one(
    state={"email": {
        "from": "security@paypa1-support.com",
        "subject": "Your account will be closed",
        "body": "URGENT: verify your identity within 24 hours at http://paypa1-support.com/verify",
    }},
    questions={
        "phishing": Noul("Is this email a phishing attempt?"),
        "kind": Choice("What kind of email is this?", {
            "phishing": "Tries to steal credentials, money or access",
            "spam": "Unwanted marketing",
            "legitimate": "A genuine message from a real sender"}),
        "pressure": Score("How much pressure does the sender put on the reader?", ["none", "some", "a lot"]),
    },
)
print(device, r.model)
print("phishing:", r.nouls["phishing"].noul)
print("kind:", r.choices["kind"].choice, r.choices["kind"].probabilities)
print("pressure:", r.scores["pressure"].score, r.scores["pressure"].probabilities)

## Interactive demo

Runs the same app as the [demo source](https://github.com/systemonedev/systemone-builder/tree/main/spaces/kenning-demo) and prints a public `gradio.live` link. Anyone with that link uses *your* Colab session until you stop it (the link expires after a week at most); nothing you enter is stored.

In [ ]:
!wget -q -O app.py https://raw.githubusercontent.com/systemonedev/systemone-builder/main/spaces/kenning-demo/app.py
import app
app.demo.launch(share=True)